# Regresion supervisada


Usa esta plantilla cuando la variable objetivo sea un número continuo.

Ejemplos: precio de vivienda, nota, consumo eléctrico, temperatura, ventas, salario.

No la uses si la salida son clases como `aprobado/suspenso` o `barato/medio/caro`. Eso ya sería clasificación.


## Qué tengo que cambiar

- `TARGET`: nombre de la columna que quiero predecir.
- Si hay columnas de texto, usa la plantilla end to end o añade `OneHotEncoder`.
- Si hay muchos valores faltantes, revisa primero la plantilla de datos faltantes.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "target"  # CAMBIAR: nombre real de la columna objetivo

df = pd.read_csv("data.csv")

# Primer vistazo: si esto ya se ve raro, no entrenes todavía.
print(df.shape)
print(df.head())
print(df[TARGET].describe())
print(df.isna().sum().sort_values(ascending=False).head(10))

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)


## Preprocesamiento seguro

Este bloque sirve aunque haya columnas numéricas y categóricas. Aprende imputación, escalado y codificación solo con `train`, porque va dentro del `Pipeline`.


In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=1.0)),  # Ridge suele ser más estable que LinearRegression pura.
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f}  ← error medio en unidades de y")
print(f"RMSE: {rmse:.4f}  ← penaliza más los errores grandes que MAE")
print(f"R2:   {r2:.4f}  ← >0.7 bueno | 0.5-0.7 regular | <0.5 malo")

import matplotlib.pyplot as plt

# Gráfica real vs predicho — indispensable en examen de regresión
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.4, s=15)
# La línea diagonal perfecta: si los puntos están sobre ella, el modelo es perfecto
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción perfecta')
plt.xlabel('Valor real')
plt.ylabel('Valor predicho')
plt.title(f'Real vs Predicho  (R²={r2:.3f})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Cross-validation sobre el conjunto de train completo
# Da una estimación más fiable que un solo split
cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
cv_mae = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')

print(f"CV R²  por fold: {cv_r2.round(4)}")
print(f"CV R²  media:    {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print()
print(f"CV MAE por fold: {(-cv_mae).round(4)}")
print(f"CV MAE media:    {(-cv_mae).mean():.4f} ± {(-cv_mae).std():.4f}")
# neg_mean_absolute_error devuelve valores negativos — el - lo invierte

# Si CV R² ≈ Test R²: el modelo generaliza bien
# Si CV R² >> Test R²: sospecha de distribución distinta en test o data leakage

## Cómo explicarlo en el examen

- `MAE`: error medio en las mismas unidades que la variable objetivo.
- `RMSE`: parecido al MAE, pero castiga más los errores grandes.
- `R2`: porcentaje aproximado de variabilidad explicado por el modelo.

Si `R2` es bajo, no digas simplemente “modelo malo”. Di que puede faltar información, haber relación no lineal, outliers o variables mal preprocesadas.


## Si falla

- Error con strings: hay columnas categóricas fuera del preprocesador.
- Error con NaN: falta imputación o la target tiene NaN.
- Métricas absurdas: revisa si `TARGET` está bien elegida.
- Mucho error: prueba `RandomForestRegressor`, transforma la target o revisa outliers.
